# Chapter 22 — Differential Forms and the Exterior Derivative

Companion notebook to Chapter 22 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

A $k$-form on an open set of $\mathbb R^n$ is stored as a function that
sends a point `x` to the array of components of shape $(n,\dots,n)$: the
entry at $(i_1,\dots,i_k)$ is the value on
$(e_{i_1},\dots,e_{i_k})$. The exterior derivative is computed by automatic
differentiation, from the formula of Theorem 22.12, which uses no
coordinates other than those of the array.

In [1]:
import itertools
import math
import warnings

import numpy as np
import torch
from torch.func import jacrev, jvp, vjp

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor


def sgn(perm):
    n_inv = sum(1 for i in range(len(perm)) for j in range(i + 1, len(perm)) if perm[i] > perm[j])
    return -1 if n_inv % 2 else 1


def antisym(t):
    # A(T) = sum over all permutations of sgn(sigma) sigma T
    k = t.dim()
    if k < 2:
        return t.clone()
    return sum(sgn(p) * t.permute(*p) for p in itertools.permutations(range(k)))


def outer(a, b):
    return torch.tensordot(a, b, dims=0)


def wedge(a, b):
    # normalization 1/(k! l!) of Definition 21.10
    return antisym(outer(a, b)) / (math.factorial(a.dim()) * math.factorial(b.dim()))


def dx(i, n):
    v = torch.zeros(n)
    v[i] = 1.0
    return v


def d(omega):
    # exterior derivative of a k-form given as x -> array (Theorem 22.12):
    # (d omega)[i0..ik] = sum_j (-1)^j  d_{i_j} omega[i0 .. (no i_j) .. ik]
    def d_omega(x):
        J = jacrev(omega)(x)          # shape (n,)*k + (n,), derivative index last
        k = J.dim() - 1
        Tn = J.movedim(-1, 0)         # derivative index first
        return sum((-1) ** j * Tn.movedim(0, j) for j in range(k + 1))
    return d_omega


def pullback(F, omega):
    # (F^* omega)(x)(v1..vk) = omega(F(x))(DF(x) v1, ..., DF(x) vk)
    def pb(x):
        J = jacrev(F)(x)
        w = omega(F(x))
        for _ in range(w.dim()):
            # contract the first slot with J and send the new index to the end
            w = torch.tensordot(w, J, dims=([0], [0]))
        return w
    return pb


def interior(v, w):
    # (i_v w)(v2..vk) = w(v, v2..vk)
    return torch.tensordot(v, w, dims=([0], [0]))


def close(a, b, tol=1e-10):
    return torch.allclose(a, b, atol=tol, rtol=tol)

2.15.0+cu132


## 1. The exterior derivative in $\mathbb R^3$

Exercise 22.1: $\alpha = yz\,dx + x^2\,dz$, $\gamma = x\,dy$,
$\beta = x\,dy\wedge dz + y^2\,dx\wedge dy$. The component of a $2$-form on
$dx_i\wedge dx_j$ is the entry $[i, j]$ of its array.

In [2]:
n = 3
e = [dx(i, n) for i in range(n)]
alpha = lambda p: p[1] * p[2] * e[0] + p[0] ** 2 * e[2]
gamma = lambda p: p[0] * e[1]
beta = lambda p: p[0] * wedge(e[1], e[2]) + p[1] ** 2 * wedge(e[0], e[1])
mu = wedge(wedge(e[0], e[1]), e[2])

p = T([0.7, -1.3, 0.4])
x, y, z = p

d_alpha = d(alpha)(p)
expected = (y - 2 * x) * wedge(e[2], e[0]) - z * wedge(e[0], e[1])
assert close(d_alpha, expected)
print("d alpha = (y - 2x) dz^dx - z dx^dy:", d_alpha[2, 0].item(), d_alpha[0, 1].item())

assert close(d(gamma)(p), wedge(e[0], e[1]))
assert close(d(beta)(p), mu)
print("d beta = dx^dy^dz: component [0,1,2] =", d(beta)(p)[0, 1, 2].item())

# d(d alpha) = 0
assert close(d(d(alpha))(p), torch.zeros(3, 3, 3))

# the signed product rule: d(alpha ^ gamma) = d alpha ^ gamma - alpha ^ d gamma
ag = lambda q: wedge(alpha(q), gamma(q))
lhs = d(ag)(p)
rhs = wedge(d(alpha)(p), gamma(p)) - wedge(alpha(p), d(gamma)(p))
assert close(lhs, rhs)
assert close(lhs, (x * y - 3 * x ** 2) * mu)
print("d(alpha ^ gamma) = (xy - 3x^2) mu:", lhs[0, 1, 2].item(), (x * y - 3 * x ** 2).item())

d alpha = (y - 2x) dz^dx - z dx^dy: -2.7 -0.4
d beta = dx^dy^dz: component [0,1,2] = 1.0
d(alpha ^ gamma) = (xy - 3x^2) mu: -2.38 -2.38


The three lines of Example 22.8: the components of $d$ on functions,
$1$-forms and $2$-forms of $\mathbb R^3$ are those of the gradient, the
curl and the divergence. And $d\circ d = 0$ on random smooth forms, in
dimension 4 as well.

In [3]:
f = lambda q: torch.sin(q[0] * q[1]) + q[2] ** 3 * q[0]
P = lambda q: torch.stack([q[1] * q[2] ** 2, torch.cos(q[0]) * q[2], q[0] * q[1] * q[2]])   # (P, Q, R)

one_form = lambda q: P(q)                               # P dx + Q dy + R dz
JP = jacrev(P)(p)                                       # JP[i, j] = d_j P_i
curl = T([JP[2, 1] - JP[1, 2], JP[0, 2] - JP[2, 0], JP[1, 0] - JP[0, 1]])
dw = d(one_form)(p)
assert close(T([dw[1, 2], dw[2, 0], dw[0, 1]]), curl)
print("components of d(P dx + Q dy + R dz) on dy^dz, dz^dx, dx^dy = curl:", curl.tolist())

two_form = lambda q: P(q)[0] * wedge(e[1], e[2]) + P(q)[1] * wedge(e[2], e[0]) + P(q)[2] * wedge(e[0], e[1])
div = JP[0, 0] + JP[1, 1] + JP[2, 2]
assert close(d(two_form)(p)[0, 1, 2], div)
print("component of d(a dy^dz + b dz^dx + c dx^dy) = divergence:", div.item())

assert close(d(d(f))(p), torch.zeros(3, 3))             # curl grad = 0
assert close(d(d(one_form))(p), torch.zeros(3, 3, 3))   # div curl = 0

# dimension 4, a 2-form with non-polynomial coefficients
n4 = 4
e4 = [dx(i, n4) for i in range(n4)]
pairs = list(itertools.combinations(range(n4), 2))
coef = lambda q: torch.stack([torch.sin(q[0] + (i + 1) * q[1]) * torch.exp(0.3 * q[2]) + q[3] ** (i % 3)
                              for i in range(len(pairs))])
omega4 = lambda q: sum(coef(q)[m] * wedge(e4[i], e4[j]) for m, (i, j) in enumerate(pairs))
q4 = torch.randn(4)
assert d(omega4)(q4).abs().max() > 0.1
assert close(d(d(omega4))(q4), torch.zeros(4, 4, 4, 4), tol=1e-9)
print("d(d omega) = 0 for a 2-form on R^4; max |d omega| =", d(omega4)(q4).abs().max().item())

components of d(P dx + Q dy + R dz) on dy^dz, dz^dx, dx^dy = curl: [-0.4848421872844885, -0.52, -0.41768707489507645]
component of d(a dy^dz + b dz^dx + c dx^dy) = divergence: -0.9099999999999999
d(d omega) = 0 for a 2-form on R^4; max |d omega| = 3.0344559931186263


## 2. Pull-back

Example 22.4 (polar coordinates), Theorem 22.5 (the determinant, without
absolute value) and Exercise 22.2.

In [4]:
polar = lambda u: torch.stack([u[0] * torch.cos(u[1]), u[0] * torch.sin(u[1])])
e2 = [dx(0, 2), dx(1, 2)]
area = lambda q: wedge(e2[0], e2[1])
alpha2 = lambda q: q[0] * e2[1] - q[1] * e2[0]            # x dy - y dx

u = T([1.7, 0.9])
r, th = u
assert close(pullback(polar, area)(u), r * wedge(e2[0], e2[1]))
assert close(pullback(polar, alpha2)(u), r ** 2 * e2[1])
print("F^*(dx^dy) = r dr^dtheta:", pullback(polar, area)(u)[0, 1].item(), " r =", r.item())
print("F^*(x dy - y dx) = r^2 dtheta:", pullback(polar, alpha2)(u).tolist())

# G(u, v) = (u^2 - v^2, 2uv): G^*(dx^dy) = 4(u^2 + v^2) du^dv
G = lambda w: torch.stack([w[0] ** 2 - w[1] ** 2, 2 * w[0] * w[1]])
w0 = T([0.6, -1.1])
assert close(pullback(G, area)(w0)[0, 1], 4 * (w0 ** 2).sum())

# top degree in R^3: F^*(g mu) = (g o F) det(DF) mu, for a map that reverses orientation at the point
A = torch.randn(3, 3)
Fmap = lambda q: A @ q + 0.3 * torch.sin(q.flip(0))
g = lambda q: 1.0 + q[0] ** 2 + q[1] * q[2]
top = lambda q: g(q) * mu
detJ = torch.linalg.det(jacrev(Fmap)(p))
assert close(pullback(Fmap, top)(p), g(Fmap(p)) * detJ * mu)
Fflip = lambda q: Fmap(q) * T([1.0, 1.0, -1.0])
detJ2 = torch.linalg.det(jacrev(Fflip)(p))
assert close(detJ2, -detJ)
assert close(pullback(Fflip, lambda q: mu)(p), -pullback(Fmap, lambda q: mu)(p))
print(f"det DF = {detJ.item():.4f}: the pull-back of mu is det(DF) mu, and changes sign with the orientation")

F^*(dx^dy) = r dr^dtheta: 1.6999999999999997  r = 1.7
F^*(x dy - y dx) = r^2 dtheta: [-2.4660561533002383e-17, 2.89]
det DF = -1.9403: the pull-back of mu is det(DF) mu, and changes sign with the orientation


### Naturality: $d(F^*\omega) = F^*(d\omega)$ (Theorem 22.9)

The map need not be injective or surjective. Below, $F:\mathbb R^2\to\mathbb
R^3$ with a $1$-form, $F:\mathbb R^3\to\mathbb R^3$ with a $2$-form, and
$F:\mathbb R^4\to\mathbb R^3$ with a $2$-form.

In [5]:
F23 = lambda w: torch.stack([w[0] * w[1], torch.sin(w[0]) + w[1] ** 2, torch.exp(0.5 * w[0] * w[1])])
lhs = d(pullback(F23, one_form))(w0)
rhs = pullback(F23, d(one_form))(w0)
assert close(lhs, rhs)
print("1-form, R^2 -> R^3:", lhs[0, 1].item(), rhs[0, 1].item())

lhs = d(pullback(Fmap, two_form))(p)
rhs = pullback(Fmap, d(two_form))(p)
assert close(lhs, rhs, tol=1e-9)
print("2-form, R^3 -> R^3:", lhs[0, 1, 2].item(), rhs[0, 1, 2].item())

B = torch.randn(3, 4)
F43 = lambda q: torch.tanh(B @ q) + torch.stack([q[0] * q[3], 0.0 * q[0], q[1] ** 2])
lhs = d(pullback(F43, two_form))(q4)
rhs = pullback(F43, d(two_form))(q4)
assert close(lhs, rhs, tol=1e-9)
print("2-form, R^4 -> R^3: max difference", (lhs - rhs).abs().max().item())

# Exercise 22.2 (1)
assert close(d(alpha2)(w0), 2 * wedge(e2[0], e2[1]))
assert close(d(pullback(polar, alpha2))(u), 2 * r * wedge(e2[0], e2[1]))

1-form, R^2 -> R^3: 0.7284707316172414 0.7284707316172414
2-form, R^3 -> R^3: -2.7793927514509917 -2.7793927514509917
2-form, R^4 -> R^3: max difference 2.498001805406602e-16


## 3. What cannot be differentiated (Remark 22.10)

The components of a covector field $\alpha$ in the polar chart are those of
$F^*\alpha$. The Jacobian array of these components is compared with the
array $J^\top[\partial\alpha]J$ that a bilinear form would give. The
difference is the symmetric term $\sum_i\alpha_i\,\partial^2F_i$, so the
antisymmetric parts agree and the symmetric parts do not.

In [6]:
a_cart = lambda q: torch.stack([q[1] ** 2 + q[0], torch.sin(q[0]) * q[1]])        # components in the chart (x, y)
a_pol = pullback(polar, a_cart)                                                   # components in the chart (r, theta)

D_cart = jacrev(a_cart)(polar(u)).T          # [i, j] = d_i alpha_j
D_pol = jacrev(a_pol)(u).T
J = jacrev(polar)(u)
as_bilinear = J.T @ D_cart @ J
H = torch.func.hessian(polar)(u)             # H[i, a, b] = d_a d_b F_i
extra = torch.einsum("i,iab->ab", a_cart(polar(u)), H)
assert close(D_pol, as_bilinear + extra)
assert close(extra, extra.T)

anti = lambda M: M - M.T
sym = lambda M: M + M.T
assert close(anti(D_pol), anti(as_bilinear))
assert not close(sym(D_pol), sym(as_bilinear), tol=1e-3)
print("antisymmetric parts agree:", anti(D_pol)[0, 1].item(), anti(as_bilinear)[0, 1].item())
print("symmetric parts differ by", (sym(D_pol) - sym(as_bilinear)).abs().max().item())

# Exercise 22.5: alpha = dx on the line, chart s with x = exp(s)
s = T(0.4)
comp_s = lambda t: torch.exp(t)              # alpha = exp(s) ds
assert close(torch.func.grad(comp_s)(s), torch.exp(s))
print("derivative of the component: 0 in the chart x,", torch.func.grad(comp_s)(s).item(), "in the chart s")

antisymmetric parts agree: -3.41447555591682 -3.41447555591682
symmetric parts differ by 9.06945818717388
derivative of the component: 0 in the chart x, 1.4918246976412703 in the chart s


## 4. The wedge with a covector, and the interior product

Lemma 22.11 and Proposition 22.18, on random alternating arrays.

In [7]:
def random_form(k, n):
    return antisym(torch.randn(*([n] * k))) if k > 0 else torch.randn(())


def evaluate(w, vs):
    for v in vs:
        w = torch.tensordot(v, w, dims=([0], [0]))
    return w


n = 5
phi = torch.randn(n)
for k in (1, 2, 3):
    eta = random_form(k, n)
    vs = [torch.randn(n) for _ in range(k + 1)]
    lhs = evaluate(wedge(phi, eta), vs)
    rhs = sum((-1) ** j * (phi @ vs[j]) * evaluate(eta, vs[:j] + vs[j + 1:]) for j in range(k + 1))
    assert close(lhs, rhs)
print("Lemma 22.11 checked for k = 1, 2, 3")

v, w = torch.randn(n), torch.randn(n)
for k, l in ((1, 1), (1, 2), (2, 1), (2, 2), (3, 1)):
    om, et = random_form(k, n), random_form(l, n)
    lhs = interior(v, wedge(om, et))
    i_om = interior(v, om)
    left = i_om * et if k == 1 else wedge(i_om, et)
    i_et = interior(v, et)
    right = om * i_et if l == 1 else wedge(om, i_et)
    assert close(lhs, left + (-1) ** k * right)
om = random_form(3, n)
assert close(interior(v, interior(w, om)), -interior(w, interior(v, om)))
assert close(interior(v, interior(v, om)), torch.zeros(n))
print("i_v is an antiderivation, and i_v i_w = - i_w i_v")

# i_E mu in R^3 and d(i_X mu) = div(X) mu  (Exercise 22.6)
n = 3
E = lambda q: q
sigma = lambda q: interior(E(q), mu)
assert close(sigma(p), x * wedge(e[1], e[2]) + y * wedge(e[2], e[0]) + z * wedge(e[0], e[1]))
assert close(d(sigma)(p), 3 * mu)
solid = lambda q: sigma(q) / q.norm() ** 3
assert close(d(solid)(p), torch.zeros(3, 3, 3))
X = lambda q: torch.stack([q[0] * q[1], torch.sin(q[2]) + q[1] ** 2, q[0] * q[2] ** 2])
divX = torch.trace(jacrev(X)(p))
assert close(d(lambda q: interior(X(q), mu))(p), divX * mu)
print("d(i_E mu) = 3 mu;  sigma / r^3 is closed;  d(i_X mu) = div(X) mu with div X =", divX.item())

Lemma 22.11 checked for k = 1, 2, 3
i_v is an antiderivation, and i_v i_w = - i_w i_v
d(i_E mu) = 3 mu;  sigma / r^3 is closed;  d(i_X mu) = div(X) mu with div X = -3.3400000000000003


## 5. Circulation around a small parallelogram (Proposition 22.13)

The integral of a $1$-form along a segment is computed by Gauss–Legendre
quadrature. The circulation $C(h)$ around the parallelogram on $hu, hv$ at
$p$, divided by $h^2$, tends to $d\omega(p)(u, v)$, with an error of order
$h$.

In [8]:
nodes, weights = np.polynomial.legendre.leggauss(40)
tq = T(0.5 * (nodes + 1.0))
wq = T(0.5 * weights)


def segment_integral(form, a, w):
    # int_0^1 form(a + t w)(w) dt
    return sum(wk * (form(a + tk * w) @ w) for tk, wk in zip(tq, wq))


def circulation(form, p0, u, v, h):
    return (segment_integral(form, p0, h * u) + segment_integral(form, p0 + h * u, h * v)
            + segment_integral(form, p0 + h * u + h * v, -h * u) + segment_integral(form, p0 + h * v, -h * v))


uu, vv = T([1.0, 0.5, -0.3]), T([0.2, -1.0, 0.8])
exact = uu @ d(one_form)(p) @ vv                     # d omega(p)(u, v)
# the same number from Theorem 22.12: (D omega u)(v) - (D omega v)(u)
Jw = jacrev(one_form)(p)
assert close(exact, (Jw @ uu) @ vv - (Jw @ vv) @ uu)

errors = []
for h in (1e-1, 1e-2, 1e-3):
    ratio = circulation(one_form, p, uu, vv, h) / h ** 2
    errors.append(abs(ratio - exact).item())
    print(f"h = {h:7.0e}   C(h)/h^2 = {ratio.item():+.8f}   d omega(u, v) = {exact.item():+.8f}   error = {errors[-1]:.2e}")
assert errors[0] < 0.5 and errors[1] < errors[0] / 5 and errors[2] < errors[1] / 5

# for an exact form the circulation vanishes for every h
df = lambda q: jacrev(f)(q)
assert abs(circulation(df, p, uu, vv, 0.5)) < 1e-12
print("circulation of df around a parallelogram of size 0.5:", circulation(df, p, uu, vv, 0.5).item())

h =   1e-01   C(h)/h^2 = +0.96599936   d omega(u, v) = +0.85817156   error = 1.08e-01
h =   1e-02   C(h)/h^2 = +0.86865297   d omega(u, v) = +0.85817156   error = 1.05e-02
h =   1e-03   C(h)/h^2 = +0.85921664   d omega(u, v) = +0.85817156   error = 1.05e-03


circulation of df around a parallelogram of size 0.5: 1.6653345369377348e-16


## 6. Closed but not exact: the angle form (Example 22.15)

$\omega = (x\,dy - y\,dx)/(x^2 + y^2)$ on the punctured plane. It is closed.
Its integral over a loop is $2\pi$ times the number of turns around the
hole, so it is not exact. On the plane without the negative half-axis it
equals $d\theta$, and the potential $K\omega$ of the Poincaré lemma with
$p_0 = (1, 0)$ is the polar angle (Exercise 22.8).

In [9]:
angle = lambda q: torch.stack([-q[1], q[0]]) / (q ** 2).sum()

for _ in range(5):
    q = torch.randn(2) * 2
    assert abs(d(angle)(q)[0, 1]) < 1e-12
print("d omega = 0 at random points")

# pull-back by the polar map is d theta
assert close(pullback(polar, angle)(u), e2[1])

ts = torch.linspace(0, 2 * math.pi, 4001)


def loop_integral(form, curve, m=1):
    # integral of c^* form over [0, 2 pi], trapezoidal rule (spectrally accurate for periodic integrands)
    vals = []
    for t in ts[:-1]:
        c, dc = jvp(curve, (t,), (T(1.0),))
        vals.append(form(c) @ dc)
    return torch.stack(vals).sum() * (2 * math.pi / (len(ts) - 1))


for m in (1, 2, -3):
    c_m = lambda t, m=m: torch.stack([torch.cos(m * t), torch.sin(m * t)])
    val = loop_integral(angle, c_m)
    assert abs(val - 2 * math.pi * m) < 1e-9
    print(f"loop turning {m:+d} times around the hole: integral = {val.item():+.8f} = 2 pi * {m}")

away = lambda t: torch.stack([2.5 + 0.6 * torch.cos(t), -2.0 + 0.6 * torch.sin(t)])
ellipse = lambda t: torch.stack([0.5 + 2.0 * torch.cos(t), 0.3 + 1.0 * torch.sin(t)])
assert abs(loop_integral(angle, away)) < 1e-9
assert abs(loop_integral(angle, ellipse) - 2 * math.pi) < 1e-9
print("loop that does not enclose the hole:", loop_integral(angle, away).item())
print("ellipse that encloses it:", loop_integral(angle, ellipse).item())

d omega = 0 at random points


loop turning +1 times around the hole: integral = +6.28318531 = 2 pi * 1


loop turning +2 times around the hole: integral = +12.56637061 = 2 pi * 2


loop turning -3 times around the hole: integral = -18.84955592 = 2 pi * -3


loop that does not enclose the hole: -9.800910479404256e-17


ellipse that encloses it: 6.283185307179586


## 7. The Poincaré lemma (Theorem 22.19)

$(K\omega)(x) = \int_0^1 t^{k-1}\, i_{x - p_0}\,\omega(p_0 + t(x - p_0))\,dt$,
by Gauss–Legendre quadrature. Automatic differentiation goes through the
quadrature, so $d(K\omega)$ is computed exactly for the quadrature rule,
and the identity $\omega = d(K\omega) + K(d\omega)$ is checked to rounding
error for smooth forms.

In [10]:
def K(form, p0):
    def K_form(xx):
        h = xx - p0
        k = form(p0).dim()
        return sum(wk * tk ** (k - 1) * interior(h, form(p0 + tk * h)) for tk, wk in zip(tq, wq))
    return K_form


origin3 = torch.zeros(3)

# Exercise 22.7 (1): a closed 1-form and its potential x^2 y + x z
a7 = lambda q: torch.stack([2 * q[0] * q[1] + q[2], q[0] ** 2, q[0]])
assert close(d(a7)(p), torch.zeros(3, 3))
assert close(K(a7, origin3)(p), x ** 2 * y + x * z)
assert close(d(K(a7, origin3))(p), a7(p))
print("K alpha = x^2 y + x z:", K(a7, origin3)(p).item(), (x ** 2 * y + x * z).item())

# Exercise 22.7 (2): omega = z dx^dy
w7 = lambda q: q[2] * wedge(e[0], e[1])
assert close(K(w7, origin3)(p), (z / 3) * (x * e[1] - y * e[0]))
assert close(K(lambda q: mu, origin3)(p), sigma(p) / 3)
assert close(d(K(w7, origin3))(p) + K(d(w7), origin3)(p), w7(p))

# omega = dx^dy on R^2
w0p = T([0.6, -1.1])
assert close(K(area, torch.zeros(2))(w0p), 0.5 * (w0p[0] * e2[1] - w0p[1] * e2[0]))

# the homotopy identity for forms that are not closed, with a centre p0 that is not the origin
p0 = T([0.3, -0.2, 0.5])
for name, form in (("1-form on R^3", one_form), ("2-form on R^3", two_form), ("3-form on R^3", top)):
    total = d(K(form, p0))(p) + (K(d(form), p0)(p) if form(p).dim() < 3 else 0.0)
    assert close(total, form(p), tol=1e-9)
    print(f"omega = d(K omega) + K(d omega), {name}: max error {(total - form(p)).abs().max().item():.1e}")

p04 = T([0.1, 0.2, -0.3, 0.4])
total = d(K(omega4, p04))(q4) + K(d(omega4), p04)(q4)
assert close(total, omega4(q4), tol=1e-9)
print(f"2-form on R^4: max error {(total - omega4(q4)).abs().max().item():.1e}")

# Exercise 22.8: on the slit plane, K(angle form) with p0 = (1, 0) is the polar angle in (-pi, pi)
for q in (T([0.5, 2.0]), T([-1.5, 2.5]), T([-0.7, -2.2]), T([3.0, -0.1])):
    val = K(angle, T([1.0, 0.0]))(q)
    assert abs(val - torch.atan2(q[1], q[0])) < 1e-8
print("K(angle form) = atan2(y, x) on the plane without the negative half-axis")

# Exercise 22.3: omega(x)(v) = b(x - p0, v)
Bm = torch.randn(3, 3)
lin = lambda q: (q - p0) @ Bm
assert close(d(lin)(p), Bm - Bm.T)
Bs = Bm + Bm.T
lin_s = lambda q: (q - p0) @ Bs
assert close(K(lin_s, p0)(p), 0.5 * (p - p0) @ Bs @ (p - p0))

K alpha = x^2 y + x z: -0.3570000000000011 -0.35699999999999993
omega = d(K omega) + K(d omega), 1-form on R^3: max error 1.9e-16


omega = d(K omega) + K(d omega), 2-form on R^3: max error 7.2e-16
omega = d(K omega) + K(d omega), 3-form on R^3: max error 2.0e-15


2-form on R^4: max error 1.1e-15
K(angle form) = atan2(y, x) on the plane without the negative half-axis


## 8. Machine learning: games

Two players descend $f$ and $g$ in their own variables. The field
$\xi = \partial_xf\,dx + \partial_yg\,dy$ is computed by automatic
differentiation, and $d\xi$ tells whether it is the differential of one
function.

In [11]:
def game_field(f, g, m):
    # xi = d_x f + d_y g for z = (x, y), x of size m
    def xi(zz):
        gf = torch.func.grad(f)(zz)
        gg = torch.func.grad(g)(zz)
        return torch.cat([gf[:m], gg[m:]])
    return xi


f_zs = lambda zz: zz[0] * zz[1]
xi_zs = game_field(f_zs, lambda zz: -f_zs(zz), 1)
z0 = T([0.6, 0.0])
assert close(xi_zs(z0), T([z0[1], -z0[0]]))
assert close(d(xi_zs)(z0)[0, 1], T(-2.0))
print("zero-sum game f = xy: d xi = -2 dx^dy")

# simultaneous gradient descent: every step multiplies the norm by sqrt(1 + eta^2)
eta = 0.4
zt = z0.clone()
for t in range(1, 21):
    zt = zt - eta * xi_zs(zt)
    assert close(zt.norm(), z0.norm() * (1 + eta ** 2) ** (t / 2))
print(f"after 20 steps the distance to the equilibrium is multiplied by {(zt.norm() / z0.norm()).item():.4f}"
      f" = (1 + eta^2)^10 = {(1 + eta ** 2) ** 10:.4f}")
print(f"angle per step = {math.degrees(math.atan(eta)):.1f} degrees, ratio per step = {math.sqrt(1 + eta ** 2):.4f}")
assert abs(math.degrees(math.atan(eta)) - 21.8) < 0.05 and abs(math.sqrt(1 + eta ** 2) - 1.077) < 5e-4

# a game with a potential: g = f + a(x) + b(y)
m = 2
f_c = lambda zz: torch.sin(zz[0] * zz[2]) + zz[1] ** 2 * zz[3] + zz[0] * zz[1]
a_x = lambda zz: zz[0] ** 3 - zz[1]
b_y = lambda zz: torch.cos(zz[2]) * zz[3]
g_c = lambda zz: f_c(zz) + a_x(zz) + b_y(zz)
xi_c = game_field(f_c, g_c, m)
z4 = torch.randn(4)
assert close(d(xi_c)(z4), torch.zeros(4, 4))
potential = lambda zz: f_c(zz) + b_y(zz)
assert close(xi_c(z4), torch.func.grad(potential)(z4))
print("g = f + a(x) + b(y): d xi = 0 and xi = d(f + b)")

# a general pair of losses: d xi has only dx_i ^ dy_j components, equal to d_x d_y (g - f)
g_gen = lambda zz: torch.cos(zz[0] * zz[3]) + zz[1] * zz[2] ** 2
xi_g = game_field(f_c, g_gen, m)
dxi = d(xi_g)(z4)
Hdiff = torch.func.hessian(lambda zz: g_gen(zz) - f_c(zz))(z4)
assert close(dxi[:m, :m], torch.zeros(2, 2)) and close(dxi[m:, m:], torch.zeros(2, 2))
assert close(dxi[:m, m:], Hdiff[:m, m:])
print("d xi = sum of d_x d_y (g - f) dx_i ^ dy_j")

zero-sum game f = xy: d xi = -2 dx^dy
after 20 steps the distance to the equilibrium is multiplied by 4.4114 = (1 + eta^2)^10 = 4.4114
angle per step = 21.8 degrees, ratio per step = 1.0770
g = f + a(x) + b(y): d xi = 0 and xi = d(f + b)
d xi = sum of d_x d_y (g - f) dx_i ^ dy_j


In [12]:
# Exercise 22.9: f = xy + eps x^2 / 2, g = -xy + eps y^2 / 2
def run(eps, eta, steps=1000):
    f_e = lambda zz: zz[0] * zz[1] + 0.5 * eps * zz[0] ** 2
    g_e = lambda zz: -zz[0] * zz[1] + 0.5 * eps * zz[1] ** 2
    xi_e = game_field(f_e, g_e, 1)
    zt = T([1.0, 0.0])
    assert close(d(xi_e)(zt)[0, 1], T(-2.0))
    for _ in range(steps):
        zt = zt - eta * xi_e(zt)
    return zt.norm().item()


eps = 0.3
threshold = 2 * eps / (1 + eps ** 2)
below, above = run(eps, 0.9 * threshold), run(eps, 1.1 * threshold)
factor = lambda eta: math.sqrt((1 - eta * eps) ** 2 + eta ** 2)
assert abs(below - factor(0.9 * threshold) ** 1000) < 1e-9 * max(1.0, below)
assert below < 1e-5 and above > 1e5
print(f"eps = {eps}: threshold step 2 eps/(1 + eps^2) = {threshold:.4f}")
print(f"  eta = 0.9 threshold: |z_1000| = {below:.3e};   eta = 1.1 threshold: |z_1000| = {above:.3e}")
assert run(0.0, 0.01, steps=2000) > 1.0
print("eps = 0: the iterates move away from 0 even with eta = 0.01:", run(0.0, 0.01, steps=2000))

eps = 0.3: threshold step 2 eps/(1 + eps^2) = 0.5505
  eta = 0.9 threshold: |z_1000| = 2.802e-07;   eta = 1.1 threshold: |z_1000| = 5.612e+07


eps = 0: the iterates move away from 0 even with eta = 0.01: 1.1051653926032317


## 9. Machine learning: is a learned score a gradient?

A network with $n$ free outputs is a general $1$-form; its exterior
derivative is the antisymmetric part of its Jacobian matrix. The gradient
of a scalar network is exact, hence closed. The defect is estimated without
forming the Jacobian matrix, with one Jacobian–vector product and one
vector–Jacobian product per pair of probes.

In [13]:
n = 6
W1, b1 = torch.randn(16, n) / math.sqrt(n), torch.randn(16)
W2, b2 = torch.randn(n, 16) / 4, torch.randn(n)
w3 = torch.randn(16) / 4

free_net = lambda q: W2 @ torch.tanh(W1 @ q + b1) + b2          # n free outputs
energy = lambda q: w3 @ torch.tanh(W1 @ q + b1)                 # scalar network
grad_net = lambda q: torch.func.grad(energy)(q)                 # its differential

xq = torch.randn(n)
J_free = jacrev(free_net)(xq)
J_grad = jacrev(grad_net)(xq)
assert close(J_grad, J_grad.T)
assert close(d(grad_net)(xq), torch.zeros(n, n))
defect = (J_free - J_free.T).norm() ** 2
assert defect > 1e-2
print(f"gradient of a scalar network: |J - J^T|_F^2 = {(J_grad - J_grad.T).norm().item() ** 2:.1e}")
print(f"network with free outputs:    |J - J^T|_F^2 = {defect.item():.4f}")

# d alpha(u, v) = v^T J u - u^T J v, from one jvp and one vjp
uu6, vv6 = torch.randn(n), torch.randn(n)
_, Ju = jvp(free_net, (xq,), (uu6,))                # J u
_, vjp_fn = vjp(free_net, xq)
uJ = vjp_fn(uu6)[0]                                 # u^T J
d_alpha_uv = vv6 @ Ju - uJ @ vv6
assert close(d_alpha_uv, uu6 @ d(free_net)(xq) @ vv6)
print("d alpha(u, v) from a jvp and a vjp:", d_alpha_uv.item())

# the mean of d alpha(u, v)^2 over Gaussian probes is |J - J^T|_F^2
gen = torch.Generator().manual_seed(1)
N = 200_000
U, V = torch.randn(N, n, generator=gen), torch.randn(N, n, generator=gen)
Bm = J_free.T - J_free
samples = torch.einsum("ni,ij,nj->n", U, Bm, V) ** 2
estimate = samples.mean()
stderr = samples.std() / math.sqrt(N)
print(f"Monte Carlo estimate {estimate.item():.4f} +- {stderr.item():.4f},  exact {defect.item():.4f}")
assert abs(estimate - defect) < 4 * stderr

gradient of a scalar network: |J - J^T|_F^2 = 2.0e-33
network with free outputs:    |J - J^T|_F^2 = 3.4130
d alpha(u, v) from a jvp and a vjp: 0.08232522882833748
Monte Carlo estimate 3.4076 +- 0.0155,  exact 3.4130


## 10. Machine learning: integrated gradients

The sum of the attributions is the integral of the exact form $dF$ along
the segment, $F(x) - F(x')$. Each attribution is the integral of
$\partial_iF\,dx_i$, which is not closed: it depends on the path and on the
coordinates (Exercise 22.10).

In [14]:
def integrated_gradients(F, x1, x0):
    h = x1 - x0
    return h * sum(wk * torch.func.grad(F)(x0 + tk * h) for tk, wk in zip(tq, wq))


net = lambda q: w3 @ torch.tanh(W1 @ q + b1) + (q ** 2).sum() * 0.1
x1, x0 = torch.randn(n), torch.randn(n)
attr = integrated_gradients(net, x1, x0)
assert close(attr.sum(), net(x1) - net(x0))
# the sum is the potential K(dF) of the Poincare lemma with p0 = x'
assert close(attr.sum(), K(lambda q: torch.func.grad(net)(q), x0)(x1))
print(f"sum of the attributions = {attr.sum().item():.10f},  F(x) - F(x') = {(net(x1) - net(x0)).item():.10f}")

prod = lambda q: q[0] * q[1]
xa, zero = T([2.0, 1.0]), torch.zeros(2)
seg = integrated_gradients(prod, xa, zero)
assert close(seg, T([1.0, 1.0]))
corner = T([2.0, 0.0])
two_legs = integrated_gradients(prod, corner, zero) + integrated_gradients(prod, xa, corner)
assert close(two_legs, T([0.0, 2.0]))
print("F = x1 x2 at (2, 1): segment", seg.tolist(), "  path along the axes", two_legs.tolist())

# in the coordinates u1 = x1 + x2, u2 = x1 - x2
Pm = 0.5 * T([[1.0, 1.0], [1.0, -1.0]])             # x = P u
prod_u = lambda uvec: prod(Pm @ uvec)
ua = torch.linalg.solve(Pm, xa)
seg_u = integrated_gradients(prod_u, ua, zero)
assert close(ua, T([3.0, 1.0])) and close(seg_u, T([9 / 4, -1 / 4]))
as_vector = torch.linalg.solve(Pm, seg)
as_covector = seg @ Pm
assert close(as_vector, T([2.0, 0.0])) and close(as_covector, T([1.0, 0.0]))
print("attributions to (u1, u2):", seg_u.tolist(), "  vector rule:", as_vector.tolist(),
      "  covector rule:", as_covector.tolist())
assert close(seg_u.sum(), seg.sum())

sum of the attributions = 0.5479814629,  F(x) - F(x') = 0.5479814629
F = x1 x2 at (2, 1): segment [1.0, 1.0]   path along the axes [0.0, 2.0]
attributions to (u1, u2): [2.2500000000000004, -0.24999999999999997]   vector rule: [2.0, -0.0]   covector rule: [1.0, 0.0]


## 11. Densities as forms of top degree

The pull-back of $p\,dx_1\wedge dx_2$ by a diffeomorphism $g$ has the
coefficient $(p\circ g)\det Dg$: the density of the change of variables
theorem when $\det Dg > 0$, and its opposite when $g$ reverses orientation.

In [15]:
gauss = lambda q: torch.exp(-0.5 * (q ** 2).sum()) / (2 * math.pi)
density_form = lambda q: gauss(q) * wedge(e2[0], e2[1])
# an affine coupling layer: z1 kept, z2 scaled and shifted by functions of z1
coupling = lambda zz: torch.stack([zz[0], zz[1] * torch.exp(0.5 * torch.tanh(zz[0])) + zz[0] ** 2])
zq = T([0.4, -0.9])
detg = torch.linalg.det(jacrev(coupling)(zq))
coef = pullback(coupling, density_form)(zq)[0, 1]
assert detg > 0 and close(coef, gauss(coupling(zq)) * detg.abs())
swap = lambda zz: coupling(zz).flip(0)
coef_swap = pullback(swap, density_form)(zq)[0, 1]
assert close(coef_swap, -coef)
print(f"coupling layer: det = {detg.item():.4f} > 0, coefficient of the pulled-back form = density = {coef.item():.6f}")
print(f"composed with a swap of the coordinates: coefficient = {coef_swap.item():.6f}")

coupling layer: det = 1.2092 > 0, coefficient of the pulled-back form = density = 0.115467
composed with a swap of the coordinates: coefficient = -0.115467


## Summary

* $d$ computed by automatic differentiation from the coordinate-free formula
  of Theorem 22.12 reproduces gradient, curl and divergence in
  $\mathbb R^3$, obeys the signed product rule and $d\circ d = 0$.
* Pull-back is substitution; in top degree it multiplies by $\det DF$, with
  its sign; $d$ commutes with the pull-back by any smooth map.
* The symmetric part of the derivative of a covector field depends on the
  chart, the antisymmetric part does not.
* The circulation of a $1$-form around a small parallelogram is
  $h^2\,d\omega(u, v)$ up to $o(h^2)$.
* The angle form is closed and integrates to $2\pi$ per turn around the
  hole; on a star-shaped domain $K\omega$ is a potential, and
  $\omega = d(K\omega) + K(d\omega)$ in general.
* The field of a zero-sum game is not closed and simultaneous descent
  spirals out; a network with free outputs is not a gradient; integrated
  gradients sum to $F(x) - F(x')$ but each one depends on the path and on
  the coordinates.